In [ ]:
from project_config import input_path, output_path, table_path, require_columns, BOOTSTRAP_RESAMPLES, RANDOM_SEED


In [ ]:
#Author Alex J. Yang, alexjieyang@outlook.com
import numpy as np
import pandas as pd
import itertools
import scipy.stats
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from tqdm import tqdm

from switch_scores import author_switch, groupby_count


In [ ]:
import gc

# switch

In [ ]:
pub2author = pd.read_csv(input_path('Author_Paper_1stlast.csv.zip') , usecols=['AuthorID','PaperID','Year'])
pub2author = pub2author.rename(columns={'PaperID': 'PublicationId', 'AuthorID': 'AuthorId'})
require_columns(pub2author, ['PublicationId', 'AuthorId', 'Year'])
pub2author = pub2author.drop_duplicates(['PublicationId', 'AuthorId'])

In [ ]:
pub2ref = pd.read_csv(input_path('SciSciNet_Paper_References.csv.zip'))
pub2ref.columns = ['CitingPublicationId', 'CitedPublicationId']

In [ ]:
pub2ref = pub2ref[pub2ref['CitingPublicationId'].isin(pub2author['PublicationId'].values)]
pub2ref.dropna(inplace = True)

In [ ]:
pub2j = pd.read_csv(input_path('SciSciNet_Papers.zip') , sep='\t', usecols=['PaperID','JournalID'])
pub2ref['CitedJournalId'] = pub2ref['CitedPublicationId'].map(pub2j.set_index('PaperID')['JournalID'])
del pub2j
pub2ref.dropna(inplace = True)
gc.collect()

In [ ]:
previous_k=None
year_window=3
pub2refjournalcounts = groupby_count(pub2ref, ['CitingPublicationId', 'CitedJournalId'], 
                                          'CitedPublicationId', count_unique=True)
pub2refjournalcounts.rename(columns={'CitedPublicationIdCount':'CitedJournalCount'}, inplace=True)

pub2refjournalcounts = pub2refjournalcounts.rename(columns={'CitingPublicationId': 'PublicationId'})

In [ ]:
pa_refs = pd.merge(pub2author, pub2refjournalcounts, how='left', on = 'PublicationId')

In [ ]:
# pa_refs.dropna(inplace=True)
# pa_refs['CitedJournalId'] = pa_refs['CitedJournalId'].astype(int)
pa_refs.sort_values(by=['AuthorId', 'Year', 'PublicationId', 'CitedJournalId'], inplace=True)
pa_refs.reset_index(drop=True, inplace=True)

In [ ]:
score_frames = []
for author_id, author_rows in pa_refs.groupby('AuthorId', sort=False):
    scores = author_switch(author_rows, previous_k=previous_k, year_window=year_window)
    scores.insert(0, 'AuthorId', author_id)
    score_frames.append(scores)
pscore = (pd.concat(score_frames, ignore_index=True) if score_frames else
          pd.DataFrame(columns=['AuthorId', 'PublicationId', 'switchScore']))
pscore.to_csv(output_path('switch_scores.csv'), index=False)

## switch Functions